# Querying ALeRCE

[ALeRCE](https://alerce.science) is the Chilean-led broker. Its public API serves the **ZTF**
stream: no account, no token, no rate limit to manage — `pip install alerce` and query.

This notebook mirrors `queryBrokers.ipynb` (Lasair/LSST) so the two are directly comparable.
Three differences worth keeping in mind:

| | Lasair | ALeRCE |
|---|---|---|
| survey here | LSST | ZTF |
| MJD scale | TAI (`lastDiaSourceMjdTai`) | UTC (`lastmjd`) |
| ingest lag | ~3.7 days when last checked | near real-time |

ALeRCE is also an LSST broker, and client 2.0.0 documents a multi-survey API taking
`survey="lsst"`. That client is not on PyPI yet (latest release is 1.4.0) and the staging
endpoint it points at returns 404, so this notebook uses the stable ZTF v1 API.

In [6]:
# pip install alerce
import pandas as pd
from astropy.time import Time
from alerce.core import Alerce

client = Alerce()
client.config["LSST_API_URL"]  # https://api.alerce.online/ztf/v1/

KeyError: 'LSST_API_URL'

### What a row is

Like Lasair, `query_objects` returns one row per **object** (`oid`), not per alert: `ndet`
detections to date, first/last detection MJD, mean position, and a classification. So "alerts
from the past N days" means objects whose most recent detection falls in the window. Individual
alerts for an object come from `query_detections` / `query_lightcurve` at the bottom.

Two quirks of the classification columns, both verified against the live API:

- `class`, `classifier` and `probability` are **null unless the query passes `classifier=`**.
  They are not object properties the endpoint fills in — an unfiltered pull comes back 100%
  null even for objects with hundreds of detections and a confident classification.
- With a `classifier=` filter, an object appears **once per classifier version**, so the same
  `oid` can occupy several rows with the same class and different probabilities. `recent_objects`
  below collapses those to the highest-probability row per object.

In [2]:
def latest_detection():
    """(mjd, days_behind) for the most recently updated object in the ZTF database."""
    now = Time.now().mjd
    df = client.query_objects(lastmjd=[now - 7, now], page_size=1,
                              order_by="lastmjd", order_mode="DESC", format="pandas")
    mjd = float(df.lastmjd.iloc[0])
    return mjd, now - mjd


mjd, lag = latest_detection()
print(f"latest detection: MJD {mjd:.4f} = {Time(mjd, format='mjd').iso} ({lag:.2f} days ago)")

latest detection: MJD 61362.5589 = 2026-11-18 13:24:53.004 (1.49 days ago)


### The query

In [3]:
MAX_COUNT = 50_000  # the API reports `total` as 50001 for anything above this


def count_recent(n_days, **filters):
    """How many objects match. Returns (count, exact): count is capped at 50000."""
    now = Time.now().mjd
    res = client.query_objects(lastmjd=[now - n_days, now], page_size=1, count=True,
                               format="json", **filters)
    total = res["total"]
    return min(total, MAX_COUNT), total <= MAX_COUNT


def recent_objects(n_days, classifier=None, class_name=None, probability=None,
                   min_ndet=None, page_size=20_000, max_pages=25, verbose=True, **filters):
    """Every ZTF object with a detection in the past `n_days`, as a DataFrame.

    classifier/class_name : e.g. "lc_classifier" + "SNIa"; see client.query_classifiers().
    probability           : minimum probability for that class.
    min_ndet              : minimum number of detections to date.

    Pages are ordered by oid so that successive pages do not overlap.
    """
    now = Time.now().mjd
    query = dict(lastmjd=[now - n_days, now], format="pandas",
                 order_by="oid", order_mode="ASC", page_size=page_size, **filters)
    if classifier:
        query["classifier"] = classifier
    if class_name:
        query["class_name"] = class_name
    if probability is not None:
        query["probability"] = probability
    if min_ndet is not None:
        query["ndet"] = [min_ndet, None]

    frames = []
    for page in range(1, max_pages + 1):
        chunk = client.query_objects(page=page, **query)
        frames.append(chunk)
        if verbose:
            print(f"page {page}: +{len(chunk)} rows (total {sum(len(f) for f in frames)})")
        if len(chunk) < page_size:
            break
    else:
        print(f"Warning: stopped at max_pages={max_pages}; results are truncated.")

    df = pd.concat(frames, ignore_index=True)
    if df.empty:
        _, lag = latest_detection()
        print(f"Nothing in the last {n_days} days — most recent detection is {lag:.2f} days old.")
        return df

    if "probability" in df and df.probability.notna().any():
        # one row per classifier version -> keep each object's best-scoring row
        before = len(df)
        df = (df.sort_values("probability", ascending=False)
                .drop_duplicates(subset="oid", keep="first"))
        if verbose and len(df) < before:
            print(f"collapsed {before} rows -> {len(df)} objects (duplicate classifier versions)")

    df["days_ago"] = (now - df.lastmjd).round(2)
    return df.sort_values("days_ago").reset_index(drop=True)

ZTF is busy — expect ~80k objects per day, so size the pull first. A page of 20000 takes
roughly 5-8 s.

In [4]:
N_DAYS = 3
n, exact = count_recent(N_DAYS)
print(f"{'' if exact else '>'}{n} objects in the last {N_DAYS} day(s)")

>50000 objects in the last 3 day(s)


In [5]:
df = recent_objects(N_DAYS)
print(len(df), "objects")
df.head()

page 1: +20000 rows (total 20000)
page 2: +20000 rows (total 40000)
page 3: +20000 rows (total 60000)
page 4: +20000 rows (total 80000)
page 5: +20000 rows (total 100000)
page 6: +20000 rows (total 120000)
page 7: +16973 rows (total 136973)
136973 objects


,oid,ndethist,ncovhist,mjdstarthist,mjdendhist,corrected,stellar,ndet,g_r_max,g_r_max_corr,...,deltajd,meanra,meandec,sigmara,sigmadec,class,classifier,probability,step_id_corr,days_ago
0,ZTF19aachurt,780,1182,58219.280405,61362.558380,True,False,448,NaN,NaN,...,2870.058160,189.815343,-13.814041,0.001788,0.001736,None,None,None,27.6.0,1.49
1,ZTF26aciaabl,17,6877,58619.318819,61362.558947,False,False,1,None,None,...,0.000000,218.621564,15.365219,0.088151,0.085000,None,None,None,27.6.0,1.49
2,ZTF20aamdmys,40,1108,58885.482361,61362.558380,True,True,10,NaN,NaN,...,2477.076018,189.384855,-19.037153,0.027476,0.025973,None,None,None,27.6.0,1.49
3,ZTF26aciaabe,2,794,61362.557801,61362.558947,False,False,2,None,None,...,0.001146,216.099356,13.455643,0.061800,0.060104,None,None,None,27.6.0,1.49
4,ZTF26achzzhj,1,2156,61362.558380,61362.558380,False,False,1,None,None,...,0.000000,188.870396,-15.734614,0.088309,0.085000,None,None,None,27.6.0,1.49


In [ ]:
# Unfiltered, the classification columns are empty — see the note above.
print("classified rows:", df["class"].notna().sum(), "of", len(df))

To get classes, ask for a classifier. Restricting to objects with a reasonable number of
detections also keeps the single-detection flood out of the way.

In [ ]:
classified = recent_objects(N_DAYS, classifier="lc_classifier", min_ndet=10)
classified["class"].value_counts()

In [ ]:
df.to_csv(f"alerce_recent_{N_DAYS}d.csv", index=False)

### Narrower cuts

Pushing the classifier cut into the query is much cheaper than pulling everything and filtering
in pandas. `query_classifiers()` lists every model and the classes it can emit.

In [ ]:
pd.DataFrame(client.query_classifiers())[["classifier_name", "classifier_version", "classes"]]

In [ ]:
sne = recent_objects(n_days=7, classifier="lc_classifier", class_name="SNIa",
                     probability=0.5, min_ndet=5)
print(len(sne), "SNIa candidates")
sne[["oid", "meanra", "meandec", "ndet", "firstmjd", "lastmjd", "days_ago", "probability"]].head()

### Individual alerts for one object

`query_detections` gives the alert-level photometry; `query_lightcurve` adds the non-detection
upper limits; `query_probabilities` gives every classifier's full probability vector.

In [ ]:
oid = sne.oid.iloc[0]

detections = client.query_detections(oid, format="pandas")
print(oid, "->", len(detections), "detections")
detections[["mjd", "fid", "magpsf", "sigmapsf", "diffmaglim", "isdiffpos", "ra", "dec"]].head()

In [ ]:
lc = client.query_lightcurve(oid, format="json")
print({k: len(v) for k, v in lc.items()})  # detections + non_detections

probs = client.query_probabilities(oid, format="pandas")
probs[probs.ranking == 1].sort_values("probability", ascending=False).head(10)

### Other calls

```python
client.query_forced_photometry(oid, format="pandas")   # forced photometry
client.query_magstats(oid, format="pandas")            # per-band summary statistics
client.query_non_detections(oid, format="pandas")
client.catshtm_conesearch(ra, dec, radius)             # catsHTM crossmatch, radius in arcsec
client.plot_stamps(oid)                                # science / reference / difference cutouts
```

ALeRCE object pages: `https://alerce.online/object/<oid>`. Client docs:
<https://alerce.readthedocs.io/>.